# LLM Zero-Shot Baseline vs DistilBERT

**Purpose:** Compare **GPT-4o-mini** (or **demo lexicon** without API key) to fine-tuned DistilBERT on the held-out test split.

**Prerequisite:** `make llm-baseline` (writes into `evaluation.json`)

---

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('dark_background')
GOLD = '#e8c547'
BERT = '#3498db'
LLM = '#9b59b6'

ROOT = Path('..').resolve()
RESULTS = ROOT / 'artifacts' / 'results'
EVAL_JSON = RESULTS / 'evaluation.json'
RESULTS.mkdir(parents=True, exist_ok=True)

if not EVAL_JSON.is_file():
    raise FileNotFoundError(f'Missing {EVAL_JSON}. Run: make train && make evaluate')

artifact = json.loads(EVAL_JSON.read_text(encoding='utf-8'))
llm_keys = [k for k in artifact.get('baselines', {}) if k.startswith('llm')]
if not llm_keys:
    print('No LLM baseline in evaluation.json — running demo baseline...')
    subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'llm_baseline.py'), '--demo', '--max-rows', '80'],
        check=True,
        cwd=str(ROOT),
        env={**dict(__import__('os').environ), 'WANDB_MODE': 'disabled', 'MLFLOW_ENABLED': 'false'},
    )
    artifact = json.loads(EVAL_JSON.read_text(encoding='utf-8'))
    llm_keys = [k for k in artifact.get('baselines', {}) if k.startswith('llm')]

test_m = artifact['splits']['test']['metrics']
llm_key = llm_keys[0]
llm_m = artifact['baselines'][llm_key]['splits']['test']['metrics']
print('DistilBERT F1:', round(test_m['f1'], 4))
print('LLM baseline:', llm_key, 'F1:', round(llm_m['f1'], 4))

In [ ]:
rows = [
    {'Model': 'DistilBERT', 'F1': test_m['f1'], 'Accuracy': test_m['accuracy']},
    {'Model': artifact['baselines'][llm_key].get('model', llm_key), 'F1': llm_m['f1'], 'Accuracy': llm_m['accuracy']},
]
df = pd.DataFrame(rows)
display(df)
df.to_csv(RESULTS / 'llm_baseline_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4))
colors = [BERT, LLM]
ax.bar(df['Model'], df['F1'], color=colors, edgecolor=GOLD, linewidth=0.8)
ax.set_ylabel('F1 (test)')
ax.set_title('DistilBERT vs LLM zero-shot', color=GOLD)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(RESULTS / 'llm_baseline_f1.png', dpi=150, facecolor='#0d0d0d')
plt.show()